# DrunkSoberNet — Colab Training

Thin notebook — all real logic lives in `src/` (cloned from GitHub).
This notebook only handles: GPU check → mount Drive → clone repo → copy data locally → run `train.py`.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo (fresh each session)

In [ ]:
%cd /content
!rm -rf CNN_Toy_Project
!git clone https://github.com/MOSAT-2026-SUMMER/CNN_Toy_Project.git
%cd CNN_Toy_Project

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5. Copy frame data from Drive to local disk

Reading thousands of small image files directly over the Drive mount is slow (network I/O).
Copy once to local disk, then all training reads from `/content/frames` instead.

In [ ]:
DRIVE_FRAMES_DIR = "/content/drive/MyDrive/MOSAT_CNN/data/frames"
LOCAL_FRAMES_DIR = "/content/frames"

!rm -rf "$LOCAL_FRAMES_DIR"
!cp -r "$DRIVE_FRAMES_DIR" "$LOCAL_FRAMES_DIR"

# No train/val split on disk -- dataset.py splits drunk/sober internally.
!echo "drunk videos:" $(ls "$LOCAL_FRAMES_DIR/drunk" | wc -l)
!echo "sober videos:" $(ls "$LOCAL_FRAMES_DIR/sober" | wc -l)

## 6. Point train.py at the local copy

`train.py`'s `DATA_ROOT` / `FRAMES_DIR` default to a Drive path. Override it here
instead of editing the source file, so the repo stays generic for both collaborators.

In [ ]:
import src.train as train_module

train_module.FRAMES_DIR = LOCAL_FRAMES_DIR
train_module.CHECKPOINT_DIR = "/content/drive/MyDrive/MOSAT_CNN/data/checkpoints"

import os
os.makedirs(train_module.CHECKPOINT_DIR, exist_ok=True)

print("FRAMES_DIR:", train_module.FRAMES_DIR)
print("CHECKPOINT_DIR:", train_module.CHECKPOINT_DIR)

## 7. Run training

Runs Phase 1 (frozen backbone) then Phase 2 (unfreeze layer4) as defined in `train.py`.
Checkpoints save straight to Drive, so they survive if the Colab session disconnects.

In [ ]:
train_module.main()

## 8. (Optional) Quick sanity check on the saved checkpoint

In [ ]:
import torch
from src.model import DrunkSoberNet

model = DrunkSoberNet()
model.load_state_dict(torch.load(
    f"{train_module.CHECKPOINT_DIR}/best.pt", map_location="cpu"
))
model.eval()
print("Checkpoint loaded OK.")